In [1]:
import pandas as pd
test_df_check = pd.read_csv(("MEDIC_test_OOD_only.tsv"), sep='\t')
print("Labels found in file:", test_df_check['humanitarian'].unique())
print("Total rows before filtering:", len(test_df_check))

Labels found in file: ['infrastructure_and_utility_damage' 'not_humanitarian'
 'affected_injured_or_dead_people'
 'rescue_volunteering_or_donation_effort']
Total rows before filtering: 10846


In [1]:
import os
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE
import ast
from tqdm import tqdm
from collections import OrderedDict

# --- CONFIGURATION ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
DATASET_PATH = "MEDIC_test_OOD_only.tsv"
IMAGE_BASE_DIR = r"F:\research\chatbot\MEDIC" 

# =================================================================
# THE FIX: Target Classes are now in STRICT ALPHABETICAL ORDER
# This matches the standard LabelEncoder you used during training.
# =================================================================
target_classes = [
    "affected_individuals",
    "infrastructure_and_utility_damage",
    "not_humanitarian",
    "other_relevant_information",
    "rescue_volunteering_or_donation_effort"
]

# --- DATASET HANDLING ---
class HumanitarianDataset(Dataset):
    def __init__(self, tsv_path, image_dir):
        self.df = pd.read_csv(tsv_path, sep='\t')
        self.image_dir = image_dir
        
        self.df['label_clean'] = self.df['humanitarian'].apply(self._clean_label)
        self.df = self.df.dropna(subset=['label_clean', 'image_path'])
        self.df = self.df[self.df['label_clean'].isin(target_classes)].reset_index(drop=True)
        
        self.label_to_idx = {label: idx for idx, label in enumerate(target_classes)}
        self.processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32", use_fast=True)

    def _clean_label(self, label):
        try:
            if str(label).startswith('['):
                label_list = ast.literal_eval(label)
                return label_list[0] if len(label_list) > 0 else "unknown"
            return str(label)
        except:
            return str(label)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_dir, str(row['image_path']))
        text = str(row.get('tweet_text', '')) 
        try:
            image = Image.open(img_path).convert("RGB")
            inputs = self.processor(text=[text], images=image, return_tensors="pt", padding=True, truncation=True)
            return inputs['pixel_values'].squeeze(0), inputs['input_ids'].squeeze(0), inputs['attention_mask'].squeeze(0), torch.tensor(self.label_to_idx[row['label_clean']])
        except:
            return None

def collate_fn(batch):
    batch = list(filter(lambda x: x is not None, batch))
    if len(batch) == 0: return None, None, None, None
    img, txt, mask, lbl = zip(*batch)
    return torch.stack(img), torch.nn.utils.rnn.pad_sequence(txt, batch_first=True), torch.nn.utils.rnn.pad_sequence(mask, batch_first=True), torch.stack(lbl)

# --- REBUILT DYNAMIC GATED FUSION ARCHITECTURE ---
class CrisisCLIPModel(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32", use_safetensors=True)
        
        self.fusion = nn.ModuleDict({
            'text_proj': nn.Linear(512, 512),
            'image_proj': nn.Linear(512, 512),
            'gate': nn.Sequential(
                nn.Linear(1024, 256),   
                nn.ReLU(),
                nn.Linear(256, 2),      
                nn.Softmax(dim=1)       
            ),
            'layer_norm': nn.LayerNorm(512)
        })
        
        self.classifier = nn.Sequential(
            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, pixel_values, input_ids, attention_mask):
        outputs = self.clip(input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values)
        
        image_embeds = self.fusion['image_proj'](outputs.image_embeds)
        text_embeds = self.fusion['text_proj'](outputs.text_embeds)
        
        combined_features = torch.cat([image_embeds, text_embeds], dim=1)
        gate_weights = self.fusion['gate'](combined_features) 
        
        alpha_image = gate_weights[:, 0].unsqueeze(1)
        alpha_text = gate_weights[:, 1].unsqueeze(1)
        
        fused_features = (alpha_image * image_embeds) + (alpha_text * text_embeds)
        fused_features = self.fusion['layer_norm'](fused_features)
        
        logits = self.classifier(fused_features)
        return logits, fused_features

# --- EVALUATION ---
print("Loading OOD Data for Task 2: Humanitarian...")
dataset = HumanitarianDataset(DATASET_PATH, IMAGE_BASE_DIR)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True, collate_fn=collate_fn)

model = CrisisCLIPModel(num_classes=5).to(device)
weights_path = r"F:\research\chatbot\MEDIC\best_humanitarian_model.pth"

# -------------------------------------------------------------
# PEFT / LoRA STATE DICT FIXER
# -------------------------------------------------------------
print("Loading and mapping PEFT/LoRA weights...")
raw_state_dict = torch.load(weights_path, weights_only=False, map_location=device)
clean_state_dict = OrderedDict()

for k, v in raw_state_dict.items():
    name = k.replace("base_model.model.", "")
    if "lora" in name:
        continue
    name = name.replace(".base_layer.", ".")
    clean_state_dict[name] = v

model.load_state_dict(clean_state_dict, strict=False)
print("Weights loaded successfully!")
# -------------------------------------------------------------

model.eval()

all_preds = []
all_labels = []
all_probs = []
all_features = []

print(f"Testing {len(dataset)} samples...")
with torch.no_grad():
    for batch in tqdm(dataloader, desc="Inference Progress"):
        if batch[0] is None: continue
        pixel_values, input_ids, attention_mask, labels = [b.to(device, non_blocking=True) for b in batch]
        
        logits, features = model(pixel_values, input_ids, attention_mask)
        probs = torch.softmax(logits, dim=1)
        
        all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_features.extend(features.cpu().numpy())

all_labels = np.array(all_labels)
all_preds = np.array(all_preds)
all_probs = np.array(all_probs)
all_features = np.array(all_features)

# ==========================================
# GRAPH 1: TEXT REPORT & CONFUSION MATRIX
# ==========================================
print("\n=== Task 2: Humanitarian OOD Report ===")
print(classification_report(all_labels, all_preds, target_names=target_classes, zero_division=0))

cm = confusion_matrix(all_labels, all_preds, normalize='true')
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Oranges", xticklabels=target_classes, yticklabels=target_classes)
plt.title("Normalized Confusion Matrix - Humanitarian (OOD)")
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig("ood_cm_humanitarian.png", dpi=300)
plt.close()
print("Saved 1/3: ood_cm_humanitarian.png")

# ==========================================
# GRAPH 2: MULTICLASS ROC CURVE
# ==========================================
y_bin = label_binarize(all_labels, classes=[0, 1, 2, 3, 4])
n_classes = y_bin.shape[1]

plt.figure(figsize=(8, 6))
colors = ['purple', 'blue', 'orange', 'gray', 'red']
for i, color in zip(range(n_classes), colors):
    # Only plot ROC if the class exists in the test set (prevents math errors on 0 support)
    if np.sum(y_bin[:, i]) > 0:
        fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
        roc_auc = auc(fpr, tpr)
        short_name = target_classes[i].split('_')[0][:10] 
        plt.plot(fpr, tpr, color=color, lw=2, label=f'ROC {short_name} (area = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic - Humanitarian (OOD)')
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("ood_roc_humanitarian.png", dpi=300)
plt.close()
print("Saved 2/3: ood_roc_humanitarian.png")

# ==========================================
# GRAPH 3: t-SNE LATENT SPACE VISUALIZATION
# ==========================================
print("Calculating t-SNE (This usually takes 1-2 minutes)...")
sample_indices = np.random.choice(len(all_features), min(1500, len(all_features)), replace=False)
features_sampled = all_features[sample_indices]
labels_sampled = all_labels[sample_indices]

tsne = TSNE(n_components=2, random_state=42, init='pca', learning_rate='auto')
tsne_results = tsne.fit_transform(features_sampled)

plt.figure(figsize=(10, 7))
sns.scatterplot(
    x=tsne_results[:, 0], 
    y=tsne_results[:, 1],
    hue=[target_classes[label] for label in labels_sampled],
    palette=sns.color_palette("Set1", n_colors=len(np.unique(labels_sampled))),
    alpha=0.7,
    s=30
)
plt.title("t-SNE Latent Space - Humanitarian (OOD)")
plt.xlabel("TSNE-1")
plt.ylabel("TSNE-2")
plt.legend(title="Class", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig("ood_tsne_humanitarian.png", dpi=300)
plt.close()
print("Saved 3/3: ood_tsne_humanitarian.png")
print("All tasks completed successfully!")

Loading OOD Data for Task 2: Humanitarian...
Loading and mapping PEFT/LoRA weights...
Weights loaded successfully!
Testing 10325 samples...


Inference Progress: 100%|██████████| 323/323 [06:28<00:00,  1.20s/it]



=== Task 2: Humanitarian OOD Report ===
                                        precision    recall  f1-score   support

                  affected_individuals       0.00      0.00      0.00         0
     infrastructure_and_utility_damage       0.00      0.00      0.00      3916
                      not_humanitarian       0.58      0.92      0.71      6138
            other_relevant_information       0.00      0.00      0.00         0
rescue_volunteering_or_donation_effort       0.01      0.01      0.01       271

                              accuracy                           0.55     10325
                             macro avg       0.12      0.19      0.14     10325
                          weighted avg       0.34      0.55      0.42     10325

Saved 1/3: ood_cm_humanitarian.png
Saved 2/3: ood_roc_humanitarian.png
Calculating t-SNE (This usually takes 1-2 minutes)...
Saved 3/3: ood_tsne_humanitarian.png
All tasks completed successfully!
